# PolicyProof AI — verify actual human adjudication and compute private descriptive metrics (CPU)

Run this only if the masked dispute-only CSV was independently completed by a third qualified human reviewer. No training, model inference, or AI grading occurs.

Upload **exactly 7 preserved files** named: `dev48_blank.csv`, `dev48_private_mapping.json`, `reviewer_A.csv`, `reviewer_B.csv`, `dev48_disputes_blank.csv`, `dev48_disputes_private_manifest.json`, and **real completed** `dev48_adjudicator_completed.csv`. Use the **SAME** original review packet, private mapping and first two reviewer files as you used to generate the dispute packet. Do not re-run random packet generation or edit the sealed manifest to get a report to pass.

The final report is owner-only and contains model comparative metrics. **Never share it or the mapping with blinded reviewers before their decisions are locked.** Successful code validation cannot prove adjudicator identity, expertise, independence or true factual model quality.

## 1 — Prepare the current CPU code and frozen synthetic dataset

In [ ]:
from pathlib import Path
import subprocess, sys, json

repo = Path("/content/PolicyProof-AI")
if not repo.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
        str(repo)
    ], check=True)
assert (repo/"scripts/adjudicate_dev48_review.py").is_file()
subprocess.run([
    sys.executable, "-m", "scripts.prepare"
], cwd=repo, check=True)
print("Original synthetic development data regenerated; no GPU and no model weights.")

## 2 — Upload the seven preserved owner-only files

These files must be from the exact same review packet and adjudication handoff. The import refuses overwrites, missing inputs and extra files.

In [ ]:
from google.colab import files

names = {
    "dev48_blank.csv",
    "dev48_private_mapping.json",
    "reviewer_A.csv",
    "reviewer_B.csv",
    "dev48_disputes_blank.csv",
    "dev48_disputes_private_manifest.json",
    "dev48_adjudicator_completed.csv",
}
uploaded = files.upload()
if set(uploaded) != names:
    raise ValueError("Upload EXACTLY the seven named original and completed private files")
private_dir = repo / "reviews"
private_dir.mkdir(parents=True, exist_ok=True)
if private_dir.is_symlink():
    raise ValueError("Private reviews directory may not be a symlink")
for name in sorted(names):
    content = uploaded[name]
    if not 0 < len(content) < 4_000_000:
        raise ValueError("Unexpectedly empty or oversized input: " + name)
    with (private_dir/name).open("xb") as handle:
        handle.write(content)
print("All seven owner-only files imported without replacement.")

## 3 — Replay original reviewer checks and verify the separate third human assessment

The tool checks all original reviewer files against their SHA-pinned manifest, the sealed randomized A/B assignment, the exact previously blank disputed rows and all actual third-reviewer verdicts. **It only replaces fields where original reviewers disagreed**; it does not invent ratings or silently change unanimous votes.

In [ ]:
subprocess.run([
    sys.executable, "-m", "scripts.adjudicate_dev48_review",
    "resolve",
    "--reviews", "reviews/reviewer_A.csv", "reviews/reviewer_B.csv",
    "--adjudicator", "reviews/dev48_adjudicator_completed.csv",
    "--output", "reviews/dev48_adjudicated_owner_only.json"
], cwd=repo, check=True)
receipt = repo/"reviews/dev48_adjudicated_owner_only.json"
result = json.loads(receipt.read_text(encoding="utf-8"))
assert result["status"] == "ADJUDICATED_DEVELOPMENT_REVIEW_REQUIRES_INDEPENDENCE_ATTESTATION"
assert result["human_approval_inferred"] is False
assert result["adjudicator_identity_or_independence_verified"] is False
assert result["semantically_validated_model_improvement"] is False
print("Contested response rows:", result["disputed_response_rows"])
print("Contested verdict fields adjudicated:", result["adjudicated_verdict_cells"])
print("Original unanimous votes preserved:", result["original_unanimous_verdicts_preserved"])
print("Model comparison is descriptive, not an official validation.")

## 4 — Download and privately archive the reconciled report

Keep the original reviewer CSVs and third reviewer CSV unchanged. Store the owner-only JSON report along with their hashes and the private A/B mapping. Do not upload any of these files to the public repository or the blinded reviewers. This is still an **inspected 48-case synthetic development** analysis and not the independent 60-case final benchmark.

In [ ]:
files.download(str(receipt))
print("Private adjudicated report downloaded. No reviewer identity or competence was authenticated.")